In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from finder_core.settings import load_settings
import json
import pandas as pd
from IPython.display import display
parameters = json.loads(Path('config/settings.json').read_text())
ticker = 'CACT'
market_map = {'SP500': '^GSPC', 'FTSE': 'FTSE100', 'DAX': 'GDAXI', 'CAC': 'CACT', 'STOXX': 'STOXX', 'CBU0': 'CBU0.L'}
market = next(name for name, value in market_map.items() if value == ticker)
train_benchmarks = False
kind = 'NEWS'

Market = market


In [ ]:
if train_benchmarks:
    import random
    import numpy as np
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix
    from finder_core.public_outputs import sequential_thresholds
    import tensorflow as tf
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


In [ ]:
if train_benchmarks:
    DATA_DIR = Path('Benchmark/Data')
    MARKET_LIST = ['SP500', 'STOXX', 'CAC', 'DAX', 'FTSE', 'CBU0']
    SEQ_LENGTH = 21
    TEST_CUTOFF = pd.Timestamp('2025-03-01')
    LABEL_HORIZON = 1
    PURGE_GAP = SEQ_LENGTH + LABEL_HORIZON - 1
    EPOCHS = 100
    BATCH_SIZE = 32
    RECURRENT_UNITS = 40
    DENSE_UNITS = 20
    DROPOUT_1 = 0.3
    DROPOUT_2 = 0.3
    LEARNING_RATE = 0.0005
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = 0.2
    EARLY_STOP_PATIENCE = 15
    LR_PATIENCE = 4
    LR_FACTOR = 0.5
    MIN_LR = 1e-06
    SAVE_RESULTS = True
    OUTPUT_DIR = Path('Benchmark/Official') / f'{Market}_{kind}'


In [ ]:
if train_benchmarks:
    from sklearn.decomposition import PCA
    news_embeddings_df = pd.read_csv(DATA_DIR / f'{Market}_daily_news_embeddings.csv', parse_dates=['date'])
    embedding_cols = [col for col in news_embeddings_df.columns if col != 'date']
    components = 10
    X = news_embeddings_df[embedding_cols].values
    pca = PCA(n_components=components, random_state=42)
    X_pca = pca.fit_transform(X)
    pca_cols = [f'news_pca_{i}' for i in range(components)]
    news_pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=news_embeddings_df.index)
    news_pca_df['date'] = news_embeddings_df['date'].values
    print('Original shape:', X.shape)
    print('PCA shape:', X_pca.shape)
    print('Explained variance:', pca.explained_variance_ratio_.sum())


In [ ]:
if train_benchmarks:
    label_path = DATA_DIR / 'Indicators_crash' / f'{Market}_df_Crash_Label.csv'
    indicator_path = DATA_DIR / 'Indicators_crash' / f'{Market}_df_Indicators.csv'
    for required_path in (label_path, indicator_path):
        if not required_path.exists():
            raise FileNotFoundError(f'Required input file not found: {required_path.resolve()}\nSet DATA_DIR to the folder containing Data/Indicators_crash and Data/OHLCV.')
    crash_labels = pd.read_csv(label_path)
    indicators = pd.read_csv(indicator_path)
    if 'date' not in crash_labels.columns:
        raise KeyError(f"Expected a 'date' column in {label_path}")
    if 'crash_label' not in crash_labels.columns:
        raise KeyError(f"Expected a 'crash_label' column in {label_path}")
    if 'Date' not in indicators.columns:
        raise KeyError(f"Expected a 'Date' column in {indicator_path}")
    crash_labels['date'] = pd.to_datetime(crash_labels['date'], errors='coerce')
    indicators['Date'] = pd.to_datetime(indicators['Date'], errors='coerce')
    crash_labels['crash_label'] = pd.to_numeric(crash_labels['crash_label'], errors='coerce')
    crash_labels = crash_labels.dropna(subset=['date', 'crash_label']).sort_values('date').drop_duplicates(subset=['date'], keep='last').reset_index(drop=True)
    indicators = indicators.dropna(subset=['Date']).sort_values('Date').drop_duplicates(subset=['Date'], keep='last').reset_index(drop=True)
    OHLCV_df_list = []
    missing_ohlcv_files = []
    for other_market in MARKET_LIST:
        if other_market == Market:
            continue
        ohlcv_path = DATA_DIR / 'OHLCV' / f'{other_market}_df_OHLCV.csv'
        if not ohlcv_path.exists():
            missing_ohlcv_files.append(str(ohlcv_path))
            continue
        other_df = pd.read_csv(ohlcv_path)
        if 'Date' not in other_df.columns:
            raise KeyError(f"Expected a 'Date' column in {ohlcv_path}")
        other_df['Date'] = pd.to_datetime(other_df['Date'], errors='coerce')
        other_df = other_df.dropna(subset=['Date']).drop_duplicates(subset=['Date'], keep='last')
        other_df = other_df.rename(columns={col: f'{col}_{other_market}' for col in other_df.columns if col != 'Date'})
        OHLCV_df_list.append(other_df)
    if missing_ohlcv_files:
        print('Warning: auxiliary OHLCV files not found; continuing without these markets:')
        for missing_path in missing_ohlcv_files:
            print(f'  - {missing_path}')
    if OHLCV_df_list:
        OHLCV_df = OHLCV_df_list[0]
        for other_df in OHLCV_df_list[1:]:
            OHLCV_df = OHLCV_df.merge(other_df, on='Date', how='outer', validate='one_to_one')
        OHLCV_df = OHLCV_df.sort_values('Date').reset_index(drop=True)
        indicators = indicators.merge(OHLCV_df, on='Date', how='left', validate='one_to_one')
    indicators = indicators.merge(news_pca_df, left_on='Date', right_on='date', how='left')
    embedding_cols_list = [col for col in indicators.columns if col.startswith('news_pca_')]
    indicators[embedding_cols_list] = indicators[embedding_cols_list].fillna(0)
    indicators = indicators.drop(columns=['date'])
    label_metadata_columns = set(crash_labels.columns) - {'date'}
    merged_df = indicators.merge(crash_labels[['date', 'crash_label']], left_on='Date', right_on='date', how='inner', validate='one_to_one')
    merged_df = merged_df.sort_values('date').drop_duplicates(subset=['date'], keep='last').reset_index(drop=True)
    if merged_df.empty:
        raise ValueError('No rows remain after aligning indicators and crash labels. Check the dates in the CSV files.')
    print(f'Merged dataframe shape: {merged_df.shape}')
    print(f"Date range: {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
    print('Crash-label distribution:')
    print(merged_df['crash_label'].value_counts(dropna=False).sort_index())


In [ ]:
if train_benchmarks:
    exclude_columns = {'Date', 'date'} | label_metadata_columns
    feature_columns = [col for col in merged_df.columns if col not in exclude_columns]
    if not feature_columns:
        raise ValueError('No feature columns remain after excluding date and label metadata columns.')
    X_raw_all = merged_df[feature_columns].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
    X_raw_all[~np.isfinite(X_raw_all)] = np.nan
    y_raw_all = merged_df['crash_label'].to_numpy(dtype=np.int32)
    dates_raw_all = pd.to_datetime(merged_df['date']).to_numpy()
    unique_labels = set(np.unique(y_raw_all).tolist())
    if not unique_labels.issubset({0, 1}) or len(unique_labels) < 2:
        raise ValueError(f'Expected binary crash_label values 0 and 1 in the aligned dataset; found {sorted(unique_labels)}.')
    if len(X_raw_all) <= SEQ_LENGTH + PURGE_GAP + 2:
        raise ValueError('Not enough aligned rows for the configured sequence length and purge gap.')
    n_samples_total = len(X_raw_all) - SEQ_LENGTH
    sample_start_indices = np.arange(n_samples_total)
    dates_seq_all = pd.to_datetime(dates_raw_all[SEQ_LENGTH:])
    y_seq_all = y_raw_all[SEQ_LENGTH:]
    pre_test_sample_indices = np.flatnonzero(dates_seq_all < TEST_CUTOFF)
    test_sample_indices = np.flatnonzero(dates_seq_all >= TEST_CUTOFF)
    if len(test_sample_indices) == 0:
        raise ValueError(f'No sequence targets fall on or after TEST_CUTOFF={TEST_CUTOFF.date()}. Choose a cutoff within the available data range.')
    if len(pre_test_sample_indices) <= PURGE_GAP + 2:
        raise ValueError('Not enough pre-test samples remain after applying the test-boundary purge.')
    if PURGE_GAP > 0:
        development_sample_indices = pre_test_sample_indices[:-PURGE_GAP]
    else:
        development_sample_indices = pre_test_sample_indices.copy()
    if len(development_sample_indices) < 10:
        raise ValueError('Too few development samples remain after purging the final test boundary.')
    train_sample_indices = development_sample_indices.copy()
    if len(train_sample_indices) == 0:
        raise ValueError('The training set is empty after purging.')
    train_raw_end = int(train_sample_indices.max() + SEQ_LENGTH + 1)
    X_train_raw_rows = X_raw_all[1:train_raw_end]
    feature_has_train_values = np.isfinite(X_train_raw_rows).any(axis=0)
    if not feature_has_train_values.any():
        raise ValueError('All feature columns are missing in the training period.')
    retained_feature_columns = [col for (col, keep) in zip(feature_columns, feature_has_train_values) if keep]
    X_raw = X_raw_all[:, feature_has_train_values]
    train_medians = np.nanmedian(X_raw[1:train_raw_end], axis=0)
    train_medians = np.where(np.isfinite(train_medians), train_medians, 0.0)
    X_raw = np.where(np.isfinite(X_raw), X_raw, train_medians.reshape(1, -1))
    X_seq_all = np.stack([X_raw[i + 1:i + SEQ_LENGTH + 1] for i in range(n_samples_total)], axis=0).astype(np.float32)
    X_train = X_seq_all[train_sample_indices]
    y_train = y_seq_all[train_sample_indices]
    dates_train = dates_seq_all[train_sample_indices]
    training_boundary_index = test_sample_indices.min()
    evaluation = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
    evaluation_cutoffs = pd.to_datetime(evaluation.loc[evaluation.market.eq(ticker), 'information_cutoff'])
    test_sample_indices = np.flatnonzero(dates_seq_all.isin(evaluation_cutoffs))
    assert len(test_sample_indices) == len(evaluation_cutoffs)
    X_test = X_seq_all[test_sample_indices]
    y_test = y_seq_all[test_sample_indices]
    dates_test = dates_seq_all[test_sample_indices]
    sequence_scaler = StandardScaler()
    sequence_scaler.fit(X_train.reshape(-1, X_train.shape[-1]))
    
    def scale_sequences(X, scaler):
        """Transform a (samples, timesteps, features) tensor with a train-fitted scaler."""
        original_shape = X.shape
        return scaler.transform(X.reshape(-1, original_shape[-1])).reshape(original_shape).astype(np.float32)
    X_train_scaled = scale_sequences(X_train, sequence_scaler)
    X_test_scaled = scale_sequences(X_test, sequence_scaler)
    print('\nDATA SPLIT SUMMARY')
    print('=' * 72)
    print(f'Sequence length:       {SEQ_LENGTH}')
    print(f'Original features:     {len(feature_columns)}')
    print(f'Retained features:     {len(retained_feature_columns)}')
    print(f'Purge gap:             {PURGE_GAP} samples')
    print(f'Training sequences:    {X_train.shape} | positives={int(y_train.sum())}')
    print(f'Final test sequences:  {X_test.shape} | positives={int(y_test.sum())}')
    print(f'Training dates:        {pd.Timestamp(dates_train.min()).date()} -> {pd.Timestamp(dates_train.max()).date()}')
    print(f'Test dates:            {pd.Timestamp(dates_test.min()).date()} -> {pd.Timestamp(dates_test.max()).date()}')
    assert dates_train.max() < dates_test.min()
    assert len(train_sample_indices) > 0 and len(test_sample_indices) > 0
    assert train_sample_indices.max() + PURGE_GAP < training_boundary_index
    print('\nLeakage checks passed: chronological train/test partitions and purge gap are in place.')


In [ ]:
if train_benchmarks:
    def calculate_class_weights(y):
        y = np.asarray(y, dtype=int)
        n_normal = int(np.sum(y == 0))
        n_anomaly = int(np.sum(y == 1))
        if n_anomaly == 0 or n_normal == 0:
            raise ValueError(f'The training partition must contain both classes for binary training; found class counts 0={n_normal}, 1={n_anomaly}.')
        return {0: 1.0, 1: float(max(5.0, n_normal / n_anomaly))}
    
    def build_recurrent_model(cell_type, n_features):
        cell_type = cell_type.upper()
        if cell_type not in {'LSTM', 'GRU'}:
            raise ValueError("cell_type must be 'LSTM' or 'GRU'.")
        recurrent_layer = tf.keras.layers.LSTM if cell_type == 'LSTM' else tf.keras.layers.GRU
        model = tf.keras.Sequential([tf.keras.layers.Input(shape=(SEQ_LENGTH, n_features)), recurrent_layer(RECURRENT_UNITS, return_sequences=False), tf.keras.layers.Dropout(DROPOUT_1), tf.keras.layers.Dense(DENSE_UNITS, activation='relu'), tf.keras.layers.Dropout(DROPOUT_2), tf.keras.layers.Dense(1, activation='sigmoid')], name=f'finder_{cell_type.lower()}')
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA), metrics=[tf.keras.metrics.AUC(name='roc_auc'), tf.keras.metrics.AUC(name='pr_auc', curve='PR'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
        return model


In [ ]:
if train_benchmarks:
    logistic = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(penalty='l1', solver='saga', max_iter=5000, class_weight='balanced', random_state=SEED))])
    logistic.fit(X_train.reshape(len(X_train), -1), y_train)
    logistic_probability = logistic.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]
    tf.keras.backend.clear_session()
    model = build_recurrent_model('LSTM', X_train.shape[-1])
    history = model.fit(X_train_scaled, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=calculate_class_weights(y_train), shuffle=True, verbose=2)
    probability = model.predict(X_test_scaled, verbose=0).ravel()


In [ ]:
if train_benchmarks:
    prices = pd.read_csv(DATA_DIR / 'OHLCV' / f'{Market}_df_OHLCV.csv', parse_dates=['Date']).sort_values('Date')
    next_dates = pd.Series(prices.Date.shift(-1).to_numpy(), index=prices.Date)
    predictions = pd.DataFrame({'target_date': next_dates.reindex(dates_test).to_numpy(), 'original_label_date': dates_test, 'feature_end_date': pd.to_datetime(dates_raw_all)[test_sample_indices + SEQ_LENGTH], 'actual': y_test, 'probability': probability, 'logistic_probability': logistic_probability})
    assert predictions.target_date.notna().all()
    finder = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
    common_dates = set(finder.loc[finder.market.eq(ticker), 'target_date'])
    predictions = predictions[predictions.target_date.dt.strftime('%Y-%m-%d').isin(common_dates)].reset_index(drop=True)
    y_test = predictions.actual.to_numpy()
    summary = []
    for name, column in [(kind, 'probability'), ('Logistic_' + kind, 'logistic_probability')]:
        thresholds = sequential_thresholds(predictions[column], y_test, predictions.target_date, predictions.original_label_date)
        predicted = (predictions[column].to_numpy() >= thresholds).astype(int)
        prefix = '' if name == kind else 'logistic_'
        predictions[prefix + 'threshold'] = thresholds
        predictions[prefix + 'prediction'] = predicted
        tn, fp, fn, tp = map(int, confusion_matrix(y_test, predicted, labels=[0, 1]).ravel())
        summary.append({'market': Market, 'model': name, 'n': len(y_test), 'events': int(y_test.sum()), 'roc_auc': roc_auc_score(y_test, predictions[column]), 'ap': average_precision_score(y_test, predictions[column]), 'accuracy': accuracy_score(y_test, predicted), 'precision': precision_score(y_test, predicted, zero_division=0), 'recall': recall_score(y_test, predicted, zero_division=0), 'f1': f1_score(y_test, predicted, zero_division=0), 'mcc': matthews_corrcoef(y_test, predicted), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
    (OUTPUT_DIR / 'metrics.json').write_text(json.dumps(summary, indent=2) + '\n')
    display(pd.DataFrame(summary))


In [ ]:
saved = Path('Benchmark/Official') / f'{Market}_{kind}'
display(pd.DataFrame(json.loads((saved / 'metrics.json').read_text())))
display(pd.read_csv(saved / 'test_predictions.csv').head())
